# Randomness

In [42]:
import pandas as pd
import numpy as np

In [43]:
# моделируем бросок монетки 1 - орел, 0,5 - вероятность выпадения орла
# size = 10 (моделируем 10 бросков сразу)
np.random.binomial(1, 0.5, size = 10)

array([0, 0, 0, 0, 0, 0, 1, 1, 1, 0])

In [44]:
# этот же метод используем для моделирования конверсии
# где 0.03 - это и будет сама конверсия
ctr = np.random.binomial(1, 0.03, size = 10)
print(ctr)
print(f'Смоделировали, что на "лендинг" зашло 10 человек и {ctr.sum()} что-то купили')
print(f'Конверсия составила: {ctr.mean():.0%}')

[1 0 0 0 0 0 0 0 0 0]
Смоделировали, что на "лендинг" зашло 10 человек и 1 что-то купили
Конверсия составила: 10%


In [45]:
# увеличим выборку и посмотрим на фактическую конверсию
np.random.binomial(1, 0.03, size = 100).mean()

np.float64(0.01)

In [46]:
# увеличим выборку до 1000
np.random.binomial(1, 0.03, size = 1000).mean()

np.float64(0.038)

**Вывод:** Закону больших чисел. Чем больше, тем лучше. С увеличением размеров выборки, выборочное среднее приближается к истинному среднему.

**Случайность** берется тогда, когда у нас маленькая выборка. Чем меньше выборка, тем большей случайности мы подверженны.

# Метод Монте-Карло
Берем случайные числа, проводим эксперименты и смотрим, что получается.

(на примере двух групп(двух лендингов) А и Б)

In [47]:
a = np.random.binomial(1, 0.03, size = 1000).mean() # контрольная группа
b = np.random.binomial(1, 0.05, size = 1000).mean() # тестовая группа

In [48]:
a, b

(np.float64(0.037), np.float64(0.05))

## Проверим, может ли быть такое, что в первой группе конверсия будет больше, чем во второй, притом что реальная конверсия в первой группе ниже, чем во второй.

In [49]:
# смоделируем 1000 раз

n = 1000
result = []
for i in range(n):
  a = np.random.binomial(1, 0.03, size = 1000).mean() # контрольная группа
  b = np.random.binomial(1, 0.05, size = 1000).mean() # тестовая группа
  result.append([a, b])

In [50]:
df = pd.DataFrame(result, columns=['a', 'b'])

In [51]:
df

,a,b
0,0.033,0.052
1,0.028,0.035
2,0.035,0.046
3,0.033,0.049
4,0.021,0.055
...,...,...
995,0.024,0.058
996,0.033,0.052
997,0.026,0.044
998,0.039,0.046


In [52]:
# получаем результат, в которых конверсия контрольной группы больше конверсии тестовой группы
df[df['a'] > df['b']]

,a,b
53,0.047,0.044
134,0.041,0.039
239,0.035,0.033
329,0.041,0.039
424,0.047,0.031
478,0.038,0.036
511,0.038,0.031
527,0.040,0.039
723,0.040,0.030
735,0.038,0.037


# AB-test calculator
Планирование эксперимента статистического теста

с использлованием онлайн-калькулятора:
https://glebmikha.github.io/ab-test-calculator-by-gleb-mikhaylov/


**Задача:** нужно определить размер выборки (чем больше выборка, тем больше уверенность. нобольшие выборки = большие затраты)


При использловании калькулятора, мы должны изначально определить какие метрики хотим, чтобы наш статистический тест имел. Начинаем с этих метрик и считаем, сколько нужно заплаьтить, чтобы получить такие метрики. А плим мы с помощью размера выборки.

**Иными словами:** мы вбиваем в калькулятор нужные нам значения, а в результате получаем размер выборки.

**Описание калькулятора:**

- Baseline Coversion Rate - Текущая конверсия (та конверсия, которую имеем на сегодняшний день)
- Minimum Detectable Effect (MDE) - Минимальный эффект, котоырй тест способен обнаружить. Та минимальная реальная разница, которую способен обнаружить тест.  
    * реальная разница, это разница между истинными конверсиями. Если брать в пример данные лендингов, то это будет разница между конверсией контрольного лендинга = 0.03 и тестового = 0.05 , т.е. разница между ними будет = 2%. Тест обнаружит эффект, разницу между гурппами, только если она будет от 2% (тут в зависимости от цифры, которую зададим в калькуляторе)  
    * -  те, если задать калькулятору MDE - 0.01, то в нашем случае, тест сможет обнаружить конверсию в новом лендинге от 4%.
- Test to Control Group Ratio - Регулирует соотношение тестовой и контрольной группы. Группы могут быть разные по размеру. Если ставим 1, то будет 1к1 = группы равные по размеру.
- Power(Desired minimum TPR) - Мощность. Желаемый минимальный TPR (напр.0.8)
- Significance (Desired maximum FPR) - Значимость. желаемый максимальный FPR (напр. 0.05)

# Evalute Results of an A/B Test (Оценка результатов AB-теста, есть разница или нет разницы)

**Описание второй части калькулятора**

- Actual Control Group Size - размер контрольной группы
- Number of Converstions in Control Group - количество конверсий в контрольной группе
- Actual Test Group Size - размер тестовой группы
- Number of Converstions in Test Group - количество конверсий в тестовой группе

# Testing stat test

In [53]:
a = np.random.binomial(1, 0.03, size = 1484).mean() # контрольная группа
b = np.random.binomial(1, 0.05, size = 1484).mean() # тестовая группа

In [54]:
a, b

(np.float64(0.02628032345013477), np.float64(0.0545822102425876))

Используя полученные данные, можно вбить их результаты во вторую часть калькулятора, чтобы получить ответ, есть ли действительно разнрица в конверсии между этими группами(лендингами)


In [55]:
a * 1484, b * 1484

(np.float64(39.0), np.float64(81.0))

Получаем положительное решение. Разница между двумя этими группами есть. Конверсия между двумя лендингами разная.

In [56]:
# на примере броска монетки оптимизируем код
print(f'Из 10 бросков монетки орел выпадет {np.random.binomial(10, 0.5)}')

Из 10 бросков монетки орел выпадет 4


In [57]:
# аналогичным образом мжоно посчитать результаты для тестовой и контрольной групп
a = np.random.binomial(1484, 0.03) # контрольная группа
b = np.random.binomial(1484, 0.05) # тестовая группа

In [58]:
a, b

(46, 68)

Снова получаем положительное решение теста.

# Калькулятор в Python

созданим функцию, которая будет принимать на вход те же значения, которые требуются для решения результатов тестирования.

In [59]:
from statsmodels.stats.proportion import proportions_ztest

In [60]:
# создаем функцию, чтобы была возможность проводить множество виртуальных жкспериментов, и посмотреть нужные метрики аб-теста
def test (conv_a, conv_b, size_a, size_b, significance = 0.05):
  _, p_value = proportions_ztest([conv_a, conv_b],
                                 [size_a, size_b],
                                 alternative='two-sided')
  return p_value < significance

In [61]:
test(40, 65, 1484, 1484)

np.True_

# TPR(Sensitivity)
сгенерируем 1000 экспериментов, и для каждого посчитаем ответ

In [62]:
n = 1000
result = []
for _ in range(n):
  a = np.random.binomial(1484, 0.03) # контрольная группа
  b = np.random.binomial(1484, 0.05) # тестовая группа
  result.append((a,b))


In [63]:
df = pd.DataFrame(result, columns=['a', 'b'])

In [64]:
# в каждой группе зашло 1484 человек, колонки "a" и "b" показывают сколько человек купило
df

,a,b
0,42,75
1,49,72
2,53,60
3,44,79
4,44,72
...,...,...
995,41,73
996,50,72
997,37,69
998,39,66


In [65]:
# на основе полученных данных нужно посчитать есть ли разница в конверсиях между контрольной и тестовой группами
# посчитаем с помощью лямбда-функции

df['test'] = df.apply(lambda x: test(x['a'], x['b'], 1484,1484), axis=1)

In [66]:
df

,a,b,test
0,42,75,True
1,49,72,True
2,53,60,False
3,44,79,True
4,44,72,True
...,...,...,...
995,41,73,True
996,50,72,True
997,37,69,True
998,39,66,True


In [67]:
df['test'].mean()

np.float64(0.818)

# FPR

In [68]:
from tqdm.notebook import tqdm # прогресс-бар, показывает как быстро выполняется код

In [69]:
n = 10000
result = []
for _ in tqdm(range(n)):
  a = np.random.binomial(1484, 0.03)
  b = np.random.binomial(1484, 0.03) # убираем разницу между группами, теперь они одиннаковые.
  result.append((a,b))

  0%|          | 0/10000 [00:00<?, ?it/s]

In [70]:

df = pd.DataFrame(result,columns=['a','b'])

In [71]:
df

,a,b
0,43,39
1,45,37
2,39,41
3,51,44
4,56,60
...,...,...
9995,43,52
9996,49,46
9997,44,42
9998,38,41


In [72]:
from tqdm import tqdm

In [73]:
tqdm.pandas()

In [74]:
df['test'] = df.progress_apply(lambda row: test(row['a'],row['b'],1484,1484),axis=1)

100%|██████████| 10000/10000 [00:06<00:00, 1490.05it/s]


In [75]:
df

,a,b,test
0,43,39,False
1,45,37,False
2,39,41,False
3,51,44,False
4,56,60,False
...,...,...,...
9995,43,52,False
9996,49,46,False
9997,44,42,False
9998,38,41,False


In [76]:
df['test'].mean()

np.float64(0.0505)

# MDE

In [77]:
from tqdm.notebook import tqdm

In [78]:
n = 1000
result = []
for _ in tqdm(range(n)):
  a = np.random.binomial(1484,0.03)
  b = np.random.binomial(1484,0.05)
  result.append((a,b))

  0%|          | 0/1000 [00:00<?, ?it/s]

In [79]:
df = pd.DataFrame(result,columns=['a','b'])

In [80]:
from tqdm import tqdm

In [81]:
df['test'] = df.progress_apply(lambda row: test(row['a'],row['b'],1484,1484),axis=1)

100%|██████████| 1000/1000 [00:00<00:00, 4334.59it/s]


In [82]:
df['test'].mean()

np.float64(0.796)

# Evan Miller test

## TPR

In [96]:
from tqdm.notebook import tqdm

In [83]:
sample_size = 1245

In [91]:
n = 10000
result = []
for _ in tqdm(range(n)):
  a = np.random.binomial(sample_size,0.03)
  b = np.random.binomial(sample_size,0.05)
  result.append((a,b))

100%|██████████| 10000/10000 [00:00<00:00, 65639.79it/s]


In [92]:
df = pd.DataFrame(result,columns=['a','b'])

In [93]:
from tqdm import tqdm

In [94]:
df['test'] = df.progress_apply(lambda row: test(row['a'],row['b'],sample_size,sample_size),axis=1)

100%|██████████| 10000/10000 [00:02<00:00, 4619.97it/s]


In [95]:
df['test'].mean()

np.float64(0.7272)

## FPR

In [97]:
from tqdm.notebook import tqdm

In [98]:
sample_size = 1245

In [99]:
n = 10000
result = []
for _ in tqdm(range(n)):
  a = np.random.binomial(sample_size,0.03)
  b = np.random.binomial(sample_size,0.03)
  result.append((a,b))

  0%|          | 0/10000 [00:00<?, ?it/s]

In [100]:
df = pd.DataFrame(result,columns=['a','b'])

In [101]:
from tqdm import tqdm

In [102]:
df['test'] = df.progress_apply(lambda row: test(row['a'],row['b'],sample_size,sample_size),axis=1)

100%|██████████| 10000/10000 [00:02<00:00, 3689.33it/s]


In [103]:
df['test'].mean()

np.float64(0.0529)